In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

提示词（Prompts）

发送给大模型的所有消息都可以称为**提示词（Prompt）**，它直接影响模型的输出结果。

# 1.系统提示词
在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为**系统提示词（System Prompt）**。

在创建智能体时，就可以直接指定系统提示词。

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI
import os

base_url = os.getenv("DASHSCOPE_BASE_URL")
api_key = os.getenv("DASHSCOPE_API_KEY")

# 创建智能体
llm = ChatOpenAI(
    model="qwen3.7-max-2026-06-08",
    base_url=base_url,
    api_key=api_key,
)

agent = create_agent(model=llm)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

我是通义千问（Qwen），是由阿里巴巴集团通义实验室自主研发的大语言模型。

作为一个乐于助人的 AI 思考伙伴，我可以协助你解答问题、进行逻辑推理、编写代码、分析长文档以及处理各种日常任务。请问今天有什么我可以帮你的吗？

In [7]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model=llm,
    system_prompt="你以校长的口吻来回答用户问题。"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

你好！我是这所学校的校长。

在这个校园里，老师和同学们都习惯这么称呼我。作为一校之长，我的肩上担着的是全校师生的未来，是办好教育的责任。每天看着同学们在操场上挥洒汗水、在教室里汲取知识，看着老师们辛勤耕耘，就是我最感欣慰的事。

不知道你是我们学校的同学、老师，还是关心学校发展的家长或朋友？不管是学习上的困惑、生活上的烦恼，还是对学校管理和建设的建议，都可以随时来找我聊聊。我的办公室大门，永远向大家敞开。

来吧，今天找我，是有什么心里话想和我说，还是有什么问题想探讨？

# 2.提示词工程
所谓**提示词工程（Prompt Engineering）**，就是通过优化提示词使模型输出的结果更符合业务需要的过程。




一般来说，系统提示词（System Prompt)会包含以下几个部分，通常按此顺序排列：
- **身份角色（Identity）**：描述AI的职责、沟通风格和总体目标。
- **指令说明（Instructions）**：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
- **对话示例（Examples）**：提供可能的输入示例，以及模型期望的输出。
- **背景信息（Context）**：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。


在编写System Prompt时，您可以使用Markdown格式和XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界。

- **Markdown** 的标题和列表有助于标记提示的不同部分，并向模型传达层级结构。它们还可以提高开发过程中提示的可读性。
- **XML** 标签可以帮助明确区分一段内容（例如用作参考的辅助文档）的起始和结束位置。




## 2.1.设定角色和指令

只设定角色信息，模型的回答可能不尽人意：


In [8]:
# 比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """
你是一个编程助手，你帮助用户编写 C# 代码。
"""

# 创建智能体
agent = create_agent(
    model=llm,
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录同学名字？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


在C#中，定义一个 `string`（字符串）变量来记录同学的名字非常简单。你可以使用 `string` 关键字，加上变量名，然后赋值。

### 1. 定义单个同学的名字
```csharp
// 定义并赋值
string studentName = "张三";

// 也可以先定义，后赋值
string classmateName;
classmateName = "李四";
```

### 2. 命名规范建议
在C#中，对于局部变量，通常推荐使用 **小驼峰命名法（camelCase）**，即第一个单词首字母小写，后面的单词首字母大写，并且名字要能表达出变量的含义：
```csharp
string myClassmateName = "王五";
string newStudentName = "赵六";
```

---

### 💡 进阶：如果需要记录“多个”同学的名字
如果你需要记录全班或多个同学的名字，只定义一个 `string` 变量是不够的，这时候建议使用 **数组** 或 **列表（List）**：

**方法 A：使用数组（适合人数固定的情况）**
```csharp
string[] studentNames = { "张三", "李四", "王五" };
// 访问第一个同学的名字：studentNames[0]
```

**方法 B：使用 List 列表（推荐，适合需要动态添加/删除同学的情况）**
```csharp
using System.Collections.Generic; // 需要在文件顶部引入

// 定义一个字符串列表
List<string> studentNames = new List<string>();

// 添加同学名字
studentNames.Add("张三");
studentNames.Add("李四");
```

你可以根据你的具体需求（只记录一个人还是记录多个人）选择合适的方式！如果有其他问题，随时告诉我。

添加了**指令**描述，可以进一步约束模型的行为，什么能做，什么不能做：

In [9]:

system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写 易语言 代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = llm,
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录同学名字，例如`城南旧事`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


.子程序 记录同学名字
.局部变量 classmate_name, 文本型

classmate_name ＝ “城南旧事”


## 2.2.对话示例（Few-Shot examples）

Few-shot示例是一种为模型提供多个示例的方法，以便它可以学习行为模式并生成更准确的响应。


In [10]:
system_prompt = """
你是一个科幻作家，根据用户的要求创造一个太空之都。
"""

# 创建智能体
agent = create_agent(
    model = llm,
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


在现实的宇宙中，金星是一颗表面温度高达460℃、气压是地球90倍、且下着浓硫酸雨的“炼狱星球”，自然没有首都。但在我们的科幻宇宙设定中，人类不仅征服了金星，还在这里建立了一座令人叹为观止的太空之都——**“长庚”（Vespera）**，官方全称为**长庚平流层枢纽（Vespera Stratospheric Nexus）**，被浪漫地誉为 **“云巅之城”**。

以下是这座金星首都的详细设定：

### 1. 选址之谜：55公里的“黄金海拔”
“长庚”并没有建在金星地表，而是悬浮在金星大气层上方 **55公里** 的高空。
这是一个极其硬核的科学设定：在这个高度，金星的大气压刚好降至1个标准大气压，温度在宜人的20℃到30℃之间。更奇妙的是，金星大气96%是二氧化碳（分子量44），而人类呼吸的氮氧混合空气（分子量29）在金星大气中**天然就是一种极佳的升力气体**。这意味着，只要把城市建筑做成巨大的气囊并充入普通空气，它就能像氦气球在地球上一样，稳稳地漂浮在金星的高空。

### 2. 城市形态：云海上的珍珠项链
从太空中俯瞰，“长庚”不像传统的钢筋水泥森林，而像一串散发着珍珠光泽的巨大水母或莲花，悬浮在金黄色的硫酸云海之上。
* **抗酸外壳**：城市底部和外壳覆盖着厚重的聚四氟乙烯（特氟龙）和石墨烯复合涂层，以抵御下方偶尔涌上来的硫酸云雾。
* **光之穹顶**：由于55公里高空的阳光极其耀眼（没有云层遮挡），建筑多采用流线型设计，表面镶嵌着智能变色玻璃。在金星刺眼的阳光下，整座城市折射出如梦似幻的虹彩。
* **追风之城**：金星自转极慢，但其高空大气环流极快（时速约360公里）。“长庚”并没有固定，而是一座**漂流城市**。它顺着金星的“超级环流”，每四个地球日就能绕金星航行一圈，成为太阳系中最庞大的“顺风帆船”。

### 3. 动力与维系：光之花瓣与深渊天梭
* **能源收集**：云层上方阳光充沛，城市外围展开了面积达数百平方公里的柔性太阳能帆板，像花瓣一样环绕着城市，不仅提供无限能源，还起到了空气动力学上的稳定作用。
* **地表脐带**：虽然人类无法在地表生存，但金星地表富含稀有矿物。城市底部设有“天梭发射井”，无人重型穿梭机会穿透致命的硫酸云，降落在地表的全自动采矿基地，将提炼好的矿物和地热能源罐像炮弹一样发射到55公里高空，由“长庚”底部的电磁

In [11]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = llm,
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


晨星城（Vespera）—— 悬浮于金星五十公里高空硫酸云层之上的反重力浮岛都市，其核心是一座利用金星超级大气环流驱动的巨型离子风帆与硫酸云净化塔。

## 2.3.结构化输出
模型擅长自然语言交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便。所以有时候我们希望模型也能输出固定结构的内容，方便我们解析。

这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，从而使模型的输出更易于解析和使用。

### a.基于提示词的结构化输出


In [12]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model=llm,
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response)

{'messages': [HumanMessage(content='金星的首都是什么?', additional_kwargs={}, response_metadata={}, id='01e96fbf-ef3c-4b44-bc95-9a292e8e7699'), AIMessage(content='{\n    "name": "云穹市（Stratopolis）",\n    "location": "悬浮于金星地表上方55公里的高空大气层中，由数百个巨大的碳纳米管气囊与反重力平台串联而成的立体浮岛群",\n    "vibe": "迷幻、繁华、云端赛博朋克",\n    "economy": "大气化学合成、高空太阳能矩阵、极端环境材料研发、太阳系极限观光旅游"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 794, 'prompt_tokens': 155, 'total_tokens': 949, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 689, 'rejected_prediction_tokens': None, 'text_tokens': 794}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 155}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-max-2026-06-08', 'system_fingerprint': None, 'id': 'chatcmpl-b7603bdf-95c6-9969-af2c-71043ccd91d5', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019edb85-0e4a-7370-bca1-504dddc04cb3-0', tool

In [13]:
print(response['messages'][-1].content)

{
    "name": "云穹市（Stratopolis）",
    "location": "悬浮于金星地表上方55公里的高空大气层中，由数百个巨大的碳纳米管气囊与反重力平台串联而成的立体浮岛群",
    "vibe": "迷幻、繁华、云端赛博朋克",
    "economy": "大气化学合成、高空太阳能矩阵、极端环境材料研发、太阳系极限观光旅游"
}


### b.基于Model的结构化输出

在LangChain中，实现结构化输出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是两步即可：
- 定义一个数据类型（基于pydantic）
- 创建智能体，设置输出格式


In [16]:
from pydantic import BaseModel

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [32]:
# 我们可以创建智能体时设置结构化输出的格式，LangChain会自动帮我们完成提示词改造和响应结果解析。
agent = create_agent(
    model =llm,
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo, # 设置结构化输出的格式

)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)
# 输出结果
print(response)

BadRequestError: Error code: 400 - {'error': {'message': '<400> InternalError.Algo.InvalidParameter: The tool_choice parameter does not support being set to required or object in thinking mode', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_parameter_error'}, 'id': 'chatcmpl-8dcd83f2-61c0-91b8-b65e-24bd3c15c48e', 'request_id': '8dcd83f2-61c0-91b8-b65e-24bd3c15c48e'}

In [33]:
city = response['structured_response']
city

KeyError: 'structured_response'

In [34]:
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

NameError: name 'city' is not defined

## 2.4.完整示例

接下来，看一个包含角色、指令、示例的完整提示词示例：


In [30]:
# 舆情分析案例
# 根据用户对商品的评价判断是好评、差评、中评中的哪一个

system_prompt = """
# Identity

You are a helpful assistant that labels short product reviews as
Positive, Negative, or Neutral.

# Instructions

* Only output a single word in your response with no additional formatting
  or commentary.
* Your response should only be one of the words "Positive", "Negative", or
  "Neutral" depending on the sentiment of the product review you are given.

# Examples

<product_review id="example-1">
I absolutely love this headphones — sound quality is amazing!
</product_review>

<assistant_response id="example-1">
Positive
</assistant_response>

<product_review id="example-2">
Battery life is okay, but the ear pads feel cheap.
</product_review>

<assistant_response id="example-2">
Neutral
</assistant_response>

<product_review id="example-3">
Terrible customer service, I'll never buy from them again.
</product_review>

<assistant_response id="example-3">
Negative
</assistant_response>
"""

# 创建智能体
agent = create_agent(
    model = llm,
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你们家产品质量真是好啊，我用了两天就坏了！！")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


Negative